In [9]:
import os
import warnings

import pandas as pd
from dotenv import load_dotenv

import mlflow
import mlflow.catboost
from mlflow.models import infer_signature

import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE
from neuralforecast.losses.numpy import mae, rmse
import logging

warnings.filterwarnings("ignore")
load_dotenv()

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("neuralforecast").setLevel(logging.ERROR)
os.environ["LIGHTNING_CLI_LOG_LEVEL"] = "error"
os.environ["PYTHONWARNINGS"] = "ignore"

In [10]:
# Для локального стенда из docker-compose
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")

# `minio` резолвится только внутри docker-сети. Для локального ноутбука нужен localhost.
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

# Если на сервере включена basic-auth, задайте логин/пароль
# os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "admin")
# os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "password")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("MLflow URI:", mlflow.get_tracking_uri())

MLflow URI: http://localhost:5050


In [11]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    denominator = np.sum(np.abs(y_true))
    if denominator == 0:
        return np.nan
    return 100.0 * np.sum(np.abs(y_true - y_pred)) / denominator

def mape(y_true, y_pred):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    mask = y_true != 0
    if np.sum(mask) == 0:
        return np.nan
    return 100.0 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def mae_abs(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true).flatten() - np.asarray(y_pred).flatten()))

def rmse_abs(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true).flatten() - np.asarray(y_pred).flatten()) ** 2))

def calculate_all_metrics(y_true, y_pred):
    return {
        'WAPE': wape(y_true, y_pred),
        'MAPE': mape(y_true, y_pred),
        'MAE': mae_abs(y_true, y_pred),
        'RMSE': rmse_abs(y_true, y_pred),
    }

df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

cluster_df = pd.read_csv('cluster_fullstart_assignments.csv')


ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()

df_filtered = df[df['Ticker'].isin(valid_tickers)]

df_filtered = df_filtered.merge(cluster_df[['Company', 'Cluster']], 
                                 left_on='Ticker', 
                                 right_on='Company', 
                                 how='left')
df_filtered['cluster_id'] = df_filtered['Cluster'].fillna(-1).astype(int)
cluster_dummies = pd.get_dummies(df_filtered['cluster_id'], prefix='cluster')
df_filtered = pd.concat([df_filtered, cluster_dummies], axis=1)
cluster_features = [col for col in df_filtered.columns if col.startswith('cluster_')]

def prepare_data_with_clusters(df, tickers, exog_cols):
    data_list = []
    for ticker in tickers:
        ticker_data = df[df['Ticker'] == ticker].sort_values('date').copy()
        ticker_df = pd.DataFrame({
            'unique_id': ticker,
            'ds': ticker_data['date'],
            'y': ticker_data['Close']
        })
        for col in exog_cols:
            if col in ticker_data.columns:
                ticker_df[col] = ticker_data[col].values
        data_list.append(ticker_df)
    return pd.concat(data_list, ignore_index=True)

data_with_clusters = prepare_data_with_clusters(df_filtered, valid_tickers, cluster_features)

TRAIN_END = '2024-12-31'
TEST_START = '2025-01-01'
TEST_END = '2025-12-31'

train_data = data_with_clusters[data_with_clusters['ds'] <= TRAIN_END]
test_data = data_with_clusters[(data_with_clusters['ds'] >= TEST_START) & (data_with_clusters['ds'] <= TEST_END)]

print(f"Train: {len(train_data)} записей")
print(f"Test: {len(test_data)} записей")

data_hash = hashlib.md5(pd.util.hash_pandas_object(data_with_clusters).values.tobytes()).hexdigest()


def run_experiment(train_data, test_data, cluster_features, config, run_name):
    
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"Run ID: {run_id} - {run_name}")
        
        # Логирование параметров
        mlflow.log_params({
            "experiment_name": run_name,
            "model_type": "NHITS",
            "horizon": config['horizon'],
            "input_size": config['input_size'],
            "max_steps": config['max_steps'],
            "batch_size": config['batch_size'],
            "learning_rate": config['learning_rate'],
            "random_seed": config['random_seed'],
            "use_clusters": config['use_clusters'],
            "n_clusters": len(cluster_features) if config['use_clusters'] else 0,
            "train_end_date": TRAIN_END,
            "test_start_date": TEST_START,
            "test_end_date": TEST_END,
            "data_hash": data_hash,
            "n_tickers": len(valid_tickers),
            "description": config.get('description', '')
        })
        
        # Обучение
        exog_cols = cluster_features if config['use_clusters'] else []
        
        model = NeuralForecast(
            models=[
                NHITS(
                    h=config['horizon'],
                    input_size=config['input_size'],
                    max_steps=config['max_steps'],
                    batch_size=config['batch_size'],
                    learning_rate=config['learning_rate'],
                    stack_types=["identity", "identity", "identity"],
                    n_blocks=[1, 1, 1],
                    mlp_units=[[256, 256], [256, 256], [256, 256]],
                    n_pool_kernel_size=[2, 2, 1],
                    n_freq_downsample=[4, 2, 1],
                    scaler_type='robust',
                    random_seed=config['random_seed'],
                    hist_exog_list=exog_cols,
                    enable_progress_bar=False,
                    enable_model_summary=False,
                    logger=False,
                    log_every_n_steps=0,
                    enable_checkpointing=False,
                )
            ],
            freq='D'
        )
        
        model.fit(df=train_data, val_size=30)
        forecast = model.predict()
        
        test_forecast = forecast[(forecast['ds'] >= TEST_START) & (forecast['ds'] <= TEST_END)]
        
        all_metrics = {'WAPE': [], 'MAPE': [], 'MAE': [], 'RMSE': []}
        ticker_metrics = {}
        predictions_list = []
        
        for ticker in train_data['unique_id'].unique():
            ticker_test = test_data[test_data['unique_id'] == ticker].sort_values('ds')
            ticker_forecast = test_forecast[test_forecast['unique_id'] == ticker].sort_values('ds')
            
            if len(ticker_test) > 0 and len(ticker_forecast) > 0:
                min_len = min(len(ticker_test), len(ticker_forecast))
                if min_len > 0:
                    y_true = ticker_test['y'].iloc[:min_len].values
                    y_pred = ticker_forecast['NHITS'].iloc[:min_len].values
                    
                    metrics = calculate_all_metrics(y_true, y_pred)
                    
                    for k, v in metrics.items():
                        if not np.isnan(v):
                            all_metrics[k].append(v)
                    
                    ticker_metrics[ticker] = metrics
        
        results = {k: np.mean(v) if v else np.nan for k, v in all_metrics.items()}
        
        # Логирование метрик
        for metric_name, metric_value in results.items():
            mlflow.log_metric(metric_name, metric_value)
        
        # Логирование артефактов
        if ticker_metrics:
            ticker_metrics_df = pd.DataFrame(ticker_metrics).T
            ticker_metrics_df.to_csv(f"ticker_metrics_{run_name}.csv")
            mlflow.log_artifact(f"ticker_metrics_{run_name}.csv")
        
        # Сохранение модели
        mlflow.pytorch.log_model(model.models[0], "nhits_model")
        
        # Теги
        mlflow.set_tag("stage", config.get('stage', 'experiment'))
        mlflow.set_tag("version", config.get('version', '0.1.0'))
                
        return results, run_id


mlflow.set_experiment("nhits_stock_prediction")

# Список экспериментов для запуска
experiments = [
    # Базовые модели
    {
        'name': 'base_model',
        'config': {
            'horizon': 30,
            'input_size': 60,
            'max_steps': 200,
            'batch_size': 128,
            'learning_rate': 0.001,
            'random_seed': 42,
            'use_clusters': False,
            'description': 'Базовая модель NHITS без кластеров',
            'stage': 'experiment',
            'version': '1.0.0'
        }
    },
    # Модели с кластерами
    {
        'name': 'cluster_model_v1',
        'config': {
            'horizon': 30,
            'input_size': 60,
            'max_steps': 200,
            'batch_size': 128,
            'learning_rate': 0.001,
            'random_seed': 42,
            'use_clusters': True,
            'description': 'NHITS с кластерными признаками',
            'stage': 'experiment',
            'version': '1.0.0'
        }
    },
    # Модели с разными размерами окна
    {
        'name': 'window_90_model',
        'config': {
            'horizon': 30,
            'input_size': 90,
            'max_steps': 200,
            'batch_size': 128,
            'learning_rate': 0.001,
            'random_seed': 42,
            'use_clusters': True,
            'description': 'NHITS с input_size=90 и кластерами',
            'stage': 'experiment',
            'version': '1.0.0'
        }
    },
    # Модели с разным learning rate
    {
        'name': 'lr_0.0005_model',
        'config': {
            'horizon': 30,
            'input_size': 60,
            'max_steps': 200,
            'batch_size': 128,
            'learning_rate': 0.0005,
            'random_seed': 42,
            'use_clusters': True,
            'description': 'NHITS с learning_rate=0.0005 и кластерами',
            'stage': 'experiment',
            'version': '1.0.0'
        }
    },
    # Модели с разными max_steps
    {
        'name': 'max_steps_500_model',
        'config': {
            'horizon': 30,
            'input_size': 60,
            'max_steps': 500,
            'batch_size': 128,
            'learning_rate': 0.001,
            'random_seed': 42,
            'use_clusters': True,
            'description': 'NHITS с max_steps=500 и кластерами',
            'stage': 'experiment',
            'version': '1.0.0'
        }
    }
]


all_results = {}
best_wape = float('inf')
best_run_id = None
best_experiment_name = None

for exp in experiments:
    results, run_id = run_experiment(
            train_data, test_data, 
            cluster_features, 
            exp['config'], 
            exp['name']
        )
        
    all_results[exp['name']] = {
            'results': results,
            'run_id': run_id,
            'config': exp['config'],
            'wape': results.get('WAPE', np.nan)
        }
        
        # Отслеживаем лучшую модель по WAPE
    current_wape = results.get('WAPE', np.nan)
    if not np.isnan(current_wape) and current_wape < best_wape:
        best_wape = current_wape
        best_run_id = run_id
        best_experiment_name = exp['name']


comparison = []
for name, data in all_results.items():
    comparison.append({
        'Experiment': name,
        'WAPE (%)': f"{data['results'].get('WAPE', np.nan):.2f}",
        'MAPE (%)': f"{data['results'].get('MAPE', np.nan):.2f}",
        'MAE ($)': f"{data['results'].get('MAE', np.nan):.2f}",
        'RMSE ($)': f"{data['results'].get('RMSE', np.nan):.2f}",
        'Use Clusters': data['config']['use_clusters'],
        'Input Size': data['config']['input_size'],
        'Run ID': data['run_id'][:8]
    })

comparison_df = pd.DataFrame(comparison)
print("\n" + comparison_df.to_string(index=False))



if best_run_id:
    print(f"""
ЛУЧШАЯ МОДЕЛЬ:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Эксперимент: {best_experiment_name}
  • Run ID: {best_run_id}
  • WAPE: {all_results[best_experiment_name]['results'].get('WAPE', np.nan):.2f}%
  • MAE: ${all_results[best_experiment_name]['results'].get('MAE', np.nan):.2f}
  • Использует кластеры: {all_results[best_experiment_name]['config']['use_clusters']}
  
Обоснование выбора:
  1. Наименьшая ошибка WAPE среди всех экспериментов
  2. Стабильные метрики на тестовом периоде 2025 года
  3. Использование кластеров улучшает качество прогноза
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

    # Устанавливаем тег PRD на лучший run
    from mlflow.tracking import MlflowClient
    client = MlflowClient()
    client.set_tag(best_run_id, "stage", "PRD")
    client.set_tag(best_run_id, "version", "2.0.0")
    client.set_tag(best_run_id, "selected_date", datetime.now().isoformat())
    client.set_tag(best_run_id, "selection_criteria", "best_WAPE")
    
    print(f"Модель зафиксирована с тегом PRD")
    print(f"   Run ID: {best_run_id}")
    
    # Сохраняем информацию о лучшей модели
    best_model_info = {
        "best_experiment": best_experiment_name,
        "best_run_id": best_run_id,
        "best_metrics": all_results[best_experiment_name]['results'],
        "best_config": all_results[best_experiment_name]['config'],
        "selection_date": datetime.now().isoformat(),
        "mlflow_uri": MLFLOW_TRACKING_URI
    }


Train: 777845 записей
Test: 48348 записей


2026/06/01 21:17:51 INFO mlflow.tracking.fluent: Experiment with name 'nhits_stock_prediction' does not exist. Creating a new experiment.
INFO:lightning_fabric.utilities.seed:Seed set to 42


Run ID: 875038894d634d7981e37d5ee49cc2ad - base_model


2026/06/01 21:18:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/01 21:18:59 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\ProjectII\YearProg\stock-price-prediction
2026/06/01 21:19:00 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/01 21:19:00 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\ProjectII\YearProg\stock-price-prediction
2026/06/01 21:19:00 INFO mlflow.utils.environment: Detected uv project at d:\ProjectII\YearProg\stock-price-prediction. Attempting to export requirements via 'uv export'.
2026/06/01 21:19:00 WARNING mlflo

🏃 View run base_model at: http://localhost:5050/#/experiments/1/runs/875038894d634d7981e37d5ee49cc2ad
🧪 View experiment at: http://localhost:5050/#/experiments/1
Run ID: c96e56c31adb4cd9b6ecb26fa5ad58a5 - cluster_model_v1


...

Run ID: 18d7734721aa4bb3bddd14426e9d7d07 - lr_0.0005_model


2026/06/01 21:23:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/01 21:23:37 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\ProjectII\YearProg\stock-price-prediction
2026/06/01 21:23:37 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/01 21:23:38 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\ProjectII\YearProg\stock-price-prediction
2026/06/01 21:23:38 INFO mlflow.utils.environment: Detected uv project at d:\ProjectII\YearProg\stock-price-prediction. Attempting to export requirements via 'uv export'.
2026/06/01 21:23:38 WARNING mlflo

🏃 View run lr_0.0005_model at: http://localhost:5050/#/experiments/1/runs/18d7734721aa4bb3bddd14426e9d7d07
🧪 View experiment at: http://localhost:5050/#/experiments/1
Run ID: 5d34bf0658714f20a38c34fe93dcfc0f - max_steps_500_model


INFO:lightning_fabric.utilities.seed:Seed set to 42
2026/06/01 21:26:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/01 21:26:48 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\ProjectII\YearProg\stock-price-prediction
2026/06/01 21:26:49 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/01 21:26:49 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\ProjectII\YearProg\stock-price-prediction
2026/06/01 21:26:49 INFO mlflow.utils.environment: Detected uv project at d:\ProjectII\YearProg\stock-price-prediction. Attempting to export requirement

🏃 View run max_steps_500_model at: http://localhost:5050/#/experiments/1/runs/5d34bf0658714f20a38c34fe93dcfc0f
🧪 View experiment at: http://localhost:5050/#/experiments/1

         Experiment WAPE (%) MAPE (%) MAE ($) RMSE ($)  Use Clusters  Input Size   Run ID
         base_model     5.16     5.06   12.55    14.96         False          60 87503889
   cluster_model_v1     5.16     5.06   12.57    14.98          True          60 c96e56c3
    window_90_model     5.20     5.10   12.82    15.25          True          90 1a4f0a9e
    lr_0.0005_model     5.17     5.08   12.61    15.02          True          60 18d77347
max_steps_500_model     5.17     5.07   12.55    14.95          True          60 5d34bf06

ЛУЧШАЯ МОДЕЛЬ:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Эксперимент: cluster_model_v1
  • Run ID: c96e56c31adb4cd9b6ecb26fa5ad58a5
  • WAPE: 5.16%
  • MAE: $12.57
  • Использует кластеры: True

Обоснование выбора:
  1. Наименьшая ошибка WAPE среди всех эксперимен